PRIMERO CONECTAMOS CON LA API DE OPENFOODFACTS Y LISTAMOS TODOS LOS ADITIVOS QUE HAY EN LA BASE DE DATOS EN UN ARCHIVO DE TEXTO

In [17]:
import requests
import pandas as pd
import re

def generar_maestro_limpio():
    url = "https://world.openfoodfacts.org/data/taxonomies/additives.json"
    headers = {'User-Agent': 'MyDataScienceProject/1.0 (mklanibarro@gmail.com)'}
    
    print("Conectando con Open Food Facts...")
    try:
        response = requests.get(url, headers=headers, timeout=30)
        if response.status_code == 200:
            data = response.json()
            lista_limpia = []
            diccionario_descartes = {}

            # Regla: Empieza por 'e', seguido de 1 a 7 caracteres alfanuméricos
            patron = re.compile(r'^e[0-9a-z]{1,7}$', re.IGNORECASE)

            for key, value in data.items():
                # Extraemos el ID corto (ej: de 'en:e300' a 'e300')
                id_corto = key.split(':')[-1] if ':' in key else key
                nombre_en = value.get('name', {}).get('en', 'Desconocido')

                if patron.match(id_corto):
                    lista_limpia.append({
                        'id': key,
                        'codigo_e': id_corto.upper(),
                        'nombre': nombre_en
                    })
                else:
                    # Todo lo que no cumpla la regla va al diccionario de descartes
                    diccionario_descartes[key] = nombre_en
            
            # Crear DataFrame y guardar
            df = pd.DataFrame(lista_limpia)
            # USAMOS RUTA RELATIVA DIRECTA PARA EVITAR ERRORES DE CARPETA
            df.to_csv('maestro_aditivos_sucio.csv', index=False)
            
            print(f"✅ ¡Hecho! Archivo 'maestro_aditivos_sucio.csv' guardado.")
            print(f"📊 Aditivos válidos: {len(df)}")
            print(f"📦 Elementos descartados: {len(diccionario_descartes)}")
            
            return df, diccionario_descartes
        else:
            print(f"❌ Error de conexión: Código {response.status_code}")
    except Exception as e:
        print(f"❌ Error inesperado: {e}")
    
    return None, None

# Ejecución
df_final, descartes = generar_maestro_limpio()

Conectando con Open Food Facts...
✅ ¡Hecho! Archivo 'maestro_aditivos_sucio.csv' guardado.
📊 Aditivos válidos: 651
📦 Elementos descartados: 30


In [19]:
import pandas as pd

# Cargar el CSV sucio
df_maestro = pd.read_csv('maestro_aditivos_sucio.csv')

# 1. Extraer el código E (E250, E330...)
# Suponiendo que el formato es "en:e250" en la columna 'id'
df_maestro['codigo_e'] = df_maestro['id'].str.split(':').str[-1].str.upper()

# 2. Limpiar la columna nombre
# Si el nombre viene como "E333ii - Dicalcium citrate", cortamos por el guion
df_maestro['nombre_limpio'] = df_maestro['nombre'].str.split(' - ').str[-1]

# 3. Guardar el maestro reluciente
df_maestro[['id', 'codigo_e', 'nombre_limpio']].to_csv('maestro_aditivos_limpio.csv', index=False)

In [5]:
import requests
import pandas as pd
import time

def generar_dataset_masivo(maestro_path, total_objetivo=10000):
    # 1. Preparar las columnas del One-Hot Encoding
    df_maestro = pd.read_csv(maestro_path)
    columnas_aditivos = df_maestro['id'].unique().tolist()
    
    print(f"Preparando matriz para {len(columnas_aditivos)} aditivos...")
    
    productos_data = []
    headers = {'User-Agent': 'MyDataScienceProject/1.0 (mklanibarro@gmail.com)'}
    base_url = "https://world.openfoodfacts.org/cgi/search.pl"
    
    # 2. Bucle de peticiones por páginas (1000 por página)
    for page in range(1, (total_objetivo // 1000) + 1):
        params = {
            "action": "process",
            "json": 1,
            "page_size": 1000,
            "page": page,
            "fields": "product_name,additives_tags,nutriscore_grade,nova_group",
            "sort_by": "unique_scans_n" # Los más populares primero[cite: 1]
        }
        
        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=30)
            if response.status_code != 200:
                print(f"Error en página {page}. Código: {response.status_code}")
                break
            
            lote = response.json().get('products', [])
            
            for p in lote:
                # Datos básicos
                fila = {
                    'nombre_producto': p.get('product_name', 'Desconocido'),
                    'nutriscore': p.get('nutriscore_grade', 'unknown'),
                    'nova_group': p.get('nova_group', 0)
                }
                
                # Obtener aditivos del producto actual como un set para búsqueda rápida[cite: 1]
                aditivos_producto = set(p.get('additives_tags', []))
                
                # Crear las 651 columnas de 0 y 1
                for aditivo_id in columnas_aditivos:
                    fila[aditivo_id] = 1 if aditivo_id in aditivos_producto else 0
                
                productos_data.append(fila)
                
            print(f"Lote {page} completado ({len(productos_data)} productos)...")
            time.sleep(1.5) # Pausa ética para la API[cite: 1]
            
        except Exception as e:
            print(f"Error crítico en la descarga: {e}")
            break

    # 3. Convertir a DataFrame y guardar
    df_final = pd.DataFrame(productos_data)
    df_final.to_csv('dataset_10k_aditivos.csv', index=False)
    return df_final

# EJECUCIÓN (Asegúrate de que 'maestro_aditivos_limpio.csv' esté en la misma carpeta)[cite: 1]
df_entrenamiento = generar_dataset_masivo('maestro_aditivos_limpio.csv')

Preparando matriz para 651 aditivos...
Lote 1 completado (100 productos)...
Lote 2 completado (200 productos)...
Error en página 3. Código: 503


In [6]:
import requests
import pandas as pd
import time

def generar_dataset_masivo(maestro_path, total_objetivo=10000):
    # 1. Preparar las columnas del One-Hot Encoding
    df_maestro = pd.read_csv(maestro_path)
    columnas_aditivos = df_maestro['id'].unique().tolist()
    
    print(f"Preparando matriz para {len(columnas_aditivos)} aditivos...")
    
    productos_data = []
    headers = {'User-Agent': 'MyDataScienceProject/1.0 (mklanibarro@gmail.com)'}
    base_url = "https://world.openfoodfacts.org/cgi/search.pl"
    
    # Configuramos el tamaño del lote a 100
    lote_size = 100
    numero_paginas = total_objetivo // lote_size

    # 2. Bucle de peticiones por páginas (100 por página)[cite: 1]
    for page in range(1, numero_paginas + 1):
        params = {
            "action": "process",
            "json": 1,
            "page_size": lote_size,
            "page": page,
            "fields": "product_name,additives_tags,nutriscore_grade,nova_group",
            "sort_by": "unique_scans_n" 
        }
        
        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=30)
            
            # Si recibimos un 503, esperamos un poco más y reintentamos esta página
            if response.status_code == 503:
                print(f"⚠️ Servidor saturado en página {page}. Esperando 10s para reintentar...")
                time.sleep(10)
                # Opcional: podrías implementar un sistema de reintentos aquí
                continue

            if response.status_code != 200:
                print(f"Error en página {page}. Código: {response.status_code}")
                break
            
            lote = response.json().get('products', [])
            
            for p in lote:
                fila = {
                    'nombre_producto': p.get('product_name', 'Desconocido'),
                    'nutriscore': p.get('nutriscore_grade', 'unknown'),
                    'nova_group': p.get('nova_group', 0)
                }
                
                aditivos_producto = set(p.get('additives_tags', []))
                
                for aditivo_id in columnas_aditivos:
                    fila[aditivo_id] = 1 if aditivo_id in aditivos_producto else 0
                
                productos_data.append(fila)
                
            print(f"Lote {page}/{numero_paginas} completado ({len(productos_data)} productos acumulados)...")
            
            # Pausa de 2 segundos según lo solicitado[cite: 1]
            time.sleep(2) 
            
        except Exception as e:
            print(f"Error crítico en la descarga: {e}")
            break

    # 3. Convertir a DataFrame y guardar[cite: 1]
    df_final = pd.DataFrame(productos_data)
    df_final.to_csv('dataset_10k_aditivos.csv', index=False)
    print("¡Proceso finalizado! Dataset guardado como 'dataset_10k_aditivos.csv'")
    return df_final

# EJECUCIÓN[cite: 1]
df_entrenamiento = generar_dataset_masivo('maestro_aditivos_limpio.csv')

Preparando matriz para 651 aditivos...
Lote 1/100 completado (100 productos acumulados)...
⚠️ Servidor saturado en página 2. Esperando 10s para reintentar...
⚠️ Servidor saturado en página 3. Esperando 10s para reintentar...
⚠️ Servidor saturado en página 4. Esperando 10s para reintentar...
⚠️ Servidor saturado en página 5. Esperando 10s para reintentar...
⚠️ Servidor saturado en página 6. Esperando 10s para reintentar...
⚠️ Servidor saturado en página 7. Esperando 10s para reintentar...
⚠️ Servidor saturado en página 8. Esperando 10s para reintentar...
Lote 9/100 completado (197 productos acumulados)...
Lote 10/100 completado (296 productos acumulados)...
⚠️ Servidor saturado en página 11. Esperando 10s para reintentar...
⚠️ Servidor saturado en página 12. Esperando 10s para reintentar...
⚠️ Servidor saturado en página 13. Esperando 10s para reintentar...
⚠️ Servidor saturado en página 14. Esperando 10s para reintentar...
⚠️ Servidor saturado en página 15. Esperando 10s para reintenta

In [11]:
import pandas as pd

# 1. Cargar el dataset que conseguiste
df = pd.read_csv('dataset_10k_aditivos.csv') # Ajusta el nombre si es diferente

# 2. Identificar columnas de aditivos
cols_aditivos = [c for c in df.columns if c.startswith('en:e')]

# 3. Contar presencias
# Sumamos cada columna: si la suma es > 0, el aditivo existe en tu muestra
conteo = df[cols_aditivos].sum()
aditivos_reales = conteo[conteo > 0].sort_values(ascending=False)

print(f"--- RESUMEN DE LA EXTRACCIÓN ---")
print(f"Total de productos analizados: {len(df)}")
print(f"Total de columnas (aditivos posibles): {len(cols_aditivos)}")
print(f"Aditivos encontrados (con al menos un '1'): {len(aditivos_reales)}")
print(f"Aditivos 'fantasma' (siempre a cero): {len(cols_aditivos) - len(aditivos_reales)}")

print(f"\n--- TOP 10 ADITIVOS MÁS FRECUENTES ---")
print(aditivos_reales.head(10))

# 4. Densidad de la matriz
total_celdas = len(df) * len(cols_aditivos)
celdas_con_uno = aditivos_reales.sum()
print(f"\nDensidad de la matriz: {(celdas_con_uno / total_celdas):.4%}")

--- RESUMEN DE LA EXTRACCIÓN ---
Total de productos analizados: 2557
Total de columnas (aditivos posibles): 646
Aditivos encontrados (con al menos un '1'): 230
Aditivos 'fantasma' (siempre a cero): 416

--- TOP 10 ADITIVOS MÁS FRECUENTES ---
en:e322      322
en:e330      259
en:e322i     226
en:e500      194
en:e300      142
en:e471      108
en:e14xx     105
en:e503      102
en:e500ii     85
en:e202       85
dtype: int64

Densidad de la matriz: 0.2192%


In [14]:
from Bio import Entrez
import pandas as pd
import time

Entrez.email = "mklanibarro@gmail.com"

def analizar_evidencia_aditivo(codigo_e, nombre):
    # 1. Definimos los diccionarios de búsqueda (Keywords)
    terminos_negativos = "(cancer OR toxicity OR carcinogen OR hypertension OR endocrine OR inflammation OR allergen)"
    terminos_positivos = "(antioxidant OR neuroprotective OR prebiotic OR preservative OR antimicrobial)"
    terminos_modelos = "(clinical trial OR human volunteer OR in vivo OR rat OR mouse OR cell culture OR in vitro)"

    resultados = {'id': codigo_e, 'nombre': nombre}
    
    try:
        # Búsqueda General (Base)
        query_base = f"({codigo_e} OR \"{nombre}\") AND food"
        handle = Entrez.esearch(db="pubmed", term=query_base, retmax=0)
        total = int(Entrez.read(handle)["Count"])
        resultados['total_docs'] = total
        
        if total > 0:
            # Nivel 1: ¿Es Positiva o Negativa? (Polaridad)
            h_neg = Entrez.esearch(db="pubmed", term=f"{query_base} AND {terminos_negativos}", retmax=0)
            resultados['count_neg'] = int(Entrez.read(h_neg)["Count"])
            
            h_pos = Entrez.esearch(db="pubmed", term=f"{query_base} AND {terminos_positivos}", retmax=0)
            resultados['count_pos'] = int(Entrez.read(h_pos)["Count"])
            
            # Nivel 2: ¿Qué tipo de Daño? (Ejemplo con Cáncer vs Alergia)
            h_cancer = Entrez.esearch(db="pubmed", term=f"{query_base} AND (cancer OR tumor)", retmax=0)
            resultados['count_cancer'] = int(Entrez.read(h_cancer)["Count"])
            
            # Nivel 3: ¿Qué modelo biológico?
            h_human = Entrez.esearch(db="pubmed", term=f"{query_base} AND (clinical trial OR volunteer)", retmax=0)
            resultados['count_human'] = int(Entrez.read(h_human)["Count"])
            
            h_animal = Entrez.esearch(db="pubmed", term=f"{query_base} AND (rat OR mouse OR murine)", retmax=0)
            resultados['count_animal'] = int(Entrez.read(h_animal)["Count"])

        return resultados

    except Exception as e:
        print(f"Error en {codigo_e}: {e}")
        return None

# --- PRUEBA REAL CON UN ADITIVO POLÉMICO (E250 - Nitrito de Sodio) ---
print("Consultando evidencia para E250...")
prueba = analizar_evidencia_aditivo("E250", "Sodium nitrite")

if prueba:
    df_prueba = pd.DataFrame([prueba])
    # Calculamos ratios para el Árbol
    df_prueba['ratio_negatividad'] = df_prueba['count_neg'] / df_prueba['total_docs']
    df_prueba['ratio_humano'] = df_prueba['count_human'] / df_prueba['total_docs']
    
    print("\n--- RESULTADOS PARA EL ÁRBOL ---")
    print(df_prueba[['id', 'total_docs', 'count_neg', 'count_cancer', 'count_human', 'ratio_negatividad']])

Consultando evidencia para E250...

--- RESULTADOS PARA EL ÁRBOL ---
     id  total_docs  count_neg  count_cancer  count_human  ratio_negatividad
0  E250         934        320            99           16           0.342612


In [20]:
import pandas as pd
from Bio import Entrez
import time

# Configuración de identidad para NCBI
Entrez.email = "mklanibarro@gmail.com"

def ejecutar_escaneo_final():
    # 1. Cargar el maestro limpio
    df_maestro = pd.read_csv('maestro_aditivos_limpio.csv')
    
    # 2. Definir las dimensiones biológicas (Keywords)
    categorias = {
        'tox_genetica': '(cancer OR carcinogen OR mutagenic OR DNA damage)',
        'tox_endocrina': '(endocrine disruptor OR hormone OR thyroid OR estrogen)',
        'tox_neuro': '(neurotoxic OR brain OR nervous system)',
        'tox_digestiva': '(microbiota OR dysbiosis OR gut permeability OR colitis)',
        'tox_alergia': '(allergy OR anaphylaxis OR hypersensitivity)',
        'tox_organos': '(hepatotoxicity OR nephrotoxicity OR liver OR kidney)',
        'ben_antiox': '(antioxidant OR oxidative stress protection)',
        'ben_microbiol': '(antimicrobial OR preservative OR pathogen inhibition)',
        'ben_metabolico': '(metabolic health OR glycemic control OR obesity)',
        'mod_humano': '(clinical trial OR volunteer OR human study)',
        'mod_animal': '(rat OR mouse OR murine OR in vivo)',
        'mod_invitro': '(cell culture OR in vitro OR assay)'
    }

    resultados = []
    total_total = len(df_maestro)

    print(f"🚀 Iniciando escaneo científico de {total_total} aditivos...")

    for i, fila in df_maestro.iterrows():
        id_completo = fila['id']
        codigo = fila['codigo_e']
        nombre = fila['nombre_limpio']
        
        # Query combinada: Busca el código O el nombre, siempre en contexto de alimentación
        query_base = f"({codigo} OR \"{nombre}\") AND (food additive OR health)"
        
        datos_aditivo = {
            'id': id_completo,
            'codigo_e': codigo,
            'nombre': nombre
        }
        
        try:
            # Búsqueda del total de artículos
            h_base = Entrez.esearch(db="pubmed", term=query_base, retmax=0)
            total_docs = int(Entrez.read(h_base)["Count"])
            datos_aditivo['total_docs'] = total_docs
            
            if total_docs > 0:
                for cat, keywords in categorias.items():
                    query_cat = f"{query_base} AND {keywords}"
                    h_cat = Entrez.esearch(db="pubmed", term=query_cat, retmax=0)
                    count = int(Entrez.read(h_cat)["Count"])
                    
                    # Guardamos el ratio (valor entre 0 y 1) para el Árbol de Decisión
                    datos_aditivo[f'r_{cat}'] = count / total_docs
            else:
                # Si no hay ciencia, llenamos con 0 para no tener NaNs en el modelo
                for cat in categorias:
                    datos_aditivo[f'r_{cat}'] = 0.0

            resultados.append(datos_aditivo)
            
            # Control de consola cada 10 aditivos
            if (i + 1) % 10 == 0:
                print(f"✅ [{i+1}/{total_total}] Procesado: {codigo}")
            
            # Pausa de seguridad para evitar baneo de la API
            time.sleep(0.35) 

        except Exception as e:
            print(f"⚠️ Error en {codigo}: {e}. Saltando...")
            time.sleep(2)
            continue

    # 3. Guardar el Dataset de Entrenamiento Final
    df_final = pd.DataFrame(resultados)
    df_final.to_csv('dataset_entrenamiento_aditivos.csv', index=False)
    print("--- 🏁 ESCANEO COMPLETADO ---")
    return df_final

# Ejecutar
df_entrenamiento = ejecutar_escaneo_final()

🚀 Iniciando escaneo científico de 651 aditivos...
✅ [10/651] Procesado: E103
✅ [20/651] Procesado: E142
✅ [30/651] Procesado: E172II
✅ [40/651] Procesado: E426
✅ [50/651] Procesado: E165
✅ [60/651] Procesado: E150D
✅ [70/651] Procesado: E585
✅ [80/651] Procesado: E517
✅ [90/651] Procesado: E521
✅ [100/651] Procesado: E392
✅ [110/651] Procesado: E551
✅ [120/651] Procesado: E380
✅ [130/651] Procesado: E965I
✅ [140/651] Procesado: E307B
✅ [150/651] Procesado: E352II
✅ [160/651] Procesado: E494
⚠️ Error en E240: Search Backend failed: GWSearch response processing error: Request to GWSearch failed."
}. Saltando...
✅ [170/651] Procesado: E625
✅ [180/651] Procesado: E413
✅ [190/651] Procesado: E650
✅ [200/651] Procesado: E956
✅ [210/651] Procesado: E1200
✅ [220/651] Procesado: E430
✅ [230/651] Procesado: E201
✅ [240/651] Procesado: E460I
✅ [250/651] Procesado: E411
✅ [260/651] Procesado: E470AIII
✅ [270/651] Procesado: E14XX
✅ [280/651] Procesado: E470AI
✅ [290/651] Procesado: E234
✅ [300/651

In [15]:
def escaner_biologico_avanzado(codigo_e, nombre):
    # Diccionarios de keywords por columna
    categorias = {
        'tox_genetica': '(cancer OR carcinogen OR mutagenic OR DNA damage)',
        'tox_endocrina': '(endocrine disruptor OR hormone OR thyroid OR estrogen)',
        'tox_neuro': '(neurotoxic OR brain OR nervous system OR oxidative stress brain)',
        'tox_digestiva': '(microbiota OR dysbiosis OR gut permeability OR inflammation)',
        'tox_alergia': '(allergy OR anaphylaxis OR hypersensitivity OR immune response)',
        'ben_antiox': '(antioxidant OR free radical OR oxidative stress protection)',
        'ben_microbiol': '(antimicrobial OR preservative OR pathogen inhibition)',
        'ben_metabolico': '(metabolic health OR glycemic control OR cholesterol lowering)'
    }
    
    query_base = f"({codigo_e} OR \"{nombre}\") AND food"
    res = {'id': codigo_e, 'total_docs': 0}
    
    try:
        # 1. Obtener el total base
        h_base = Entrez.esearch(db="pubmed", term=query_base, retmax=0)
        total = int(Entrez.read(h_base)["Count"])
        res['total_docs'] = total
        
        if total > 0:
            # 2. Escanear cada categoría biológica
            for cat, keywords in categorias.items():
                query_cat = f"{query_base} AND {keywords}"
                h_cat = Entrez.esearch(db="pubmed", term=query_cat, retmax=0)
                count = int(Entrez.read(h_cat)["Count"])
                # Guardamos el ratio (frecuencia relativa) para el árbol
                res[f'ratio_{cat}'] = count / total
                
        return res
    except Exception as e:
        return None